# Modelo 3. ConvNeXt-Tiny con la imagen de la galaxia

**Inferencia de propiedades físicas de galaxias a partir de imágenes multibanda** · Instituto Tecnológico y de Estudios Superiores de Monterrey · Maestría en Inteligencia Artificial Aplicada · **Equipo 24** · Colaboración con el Instituto de Astronomía de la UNAM

| Integrante | Matrícula |
|---|---|
| José Florencio Maguey Peralta | A01796727 |
| Erick Alberto Bustos Cruz | A01378966 |
| Luz Copelia Minutti Pérez | A01796921 |

**Asesores:** Dr. Iván Reyes Amezcua (ITESM) · Dr. Héctor Hernández Toledo y Dr. José Antonio Vázquez Mata (IA-UNAM)

**Fase de CRISP-ML(Q):** 3, *Machine Learning Model Engineering* (Studer et al. 2021). **Avance 4:** modelos alternativos.

## 1. Planteamiento

Este cuaderno construye, ajusta y evalúa uno de los seis modelos individuales del avance: una red **ConvNeXt-Tiny** que estima la masa estelar (log M\*), la tasa de formación estelar (log SFR) y el índice D4000 a partir de la imagen de DESI en las bandas g, r y z.

ConvNeXt (Liu et al. 2022) es una ResNet50 "modernizada" paso a paso con las decisiones de diseño de los transformadores de visión, pero sin atención: filtros de 7 × 7 que se aplican a cada canal por separado, bloques invertidos, activación GELU y normalización por capa (LayerNorm). Con el mismo costo de cómputo que ResNet50, ConvNeXt-Tiny alcanza 82.1 % de exactitud en ImageNet, frente a 76.1 % de la ResNet50 original. Los autores muestran que la mayor parte de esa ganancia viene de la receta de entrenamiento moderna, y la mantenemos aquí.

Esperamos que ConvNeXt mejore a ResNet50 en nuestro problema por tres razones:

- **No depende del tamaño del lote.** ResNet50 usa normalización por lotes (BatchNorm), que funciona mal con lotes pequeños; en el modelo 1 del equipo hubo que congelar sus estadísticas. ConvNeXt usa LayerNorm, que normaliza cada imagen por separado.
- **Ve estructuras más grandes desde el inicio.** Los filtros de 7 × 7 captan mejor la estructura extendida de una galaxia, como el disco y los brazos, que están relacionados con la formación de estrellas.
- **Se ha usado con éxito en astronomía.** Zoobot, el modelo de morfología de Galaxy Zoo, ofrece versiones ConvNeXt, y se han publicado clasificadores morfológicos recientes con codificadores ConvNeXt (Fang et al. 2025).

El modelo es directamente comparable con el modelo 1 del equipo (ResNet50 multitarea con las mismas bandas y la misma pérdida, variantes F y G), porque solo cambia el extractor de la imagen: los datos, las cabezas, la pérdida y el criterio de selección son los mismos.


**Restricción de uso.** Los asesores indicaron que, en la práctica, de cada galaxia nueva solo se tendrá su imagen: el redshift no estará disponible al predecir. Por eso ningún modelo de este cuaderno recibe el redshift medido. Lo evaluamos de dos formas:

- **Solo con la imagen.**
- **En dos etapas:** primero estimamos el redshift a partir de la misma imagen (sección 5) y después lo damos al modelo junto con la imagen. Al predecir, el modelo sigue necesitando solo la imagen.

El redshift medido solo se usa como etiqueta para entrenar esa estimación y, en el diagnóstico, para revisar si el error depende de la distancia.

### Protocolo común de comparación

Los seis modelos del avance se comparan con las mismas reglas, de modo que las diferencias se puedan atribuir al algoritmo y no a cambios en los datos o en la forma de medir. Las reglas son las del modelo 1 del equipo (ResNet50 multitarea), y sus variantes F y G, que tampoco usan el redshift, son la referencia directa.

| Elemento | Regla común |
|---|---|
| Muestra | Galaxias con QCFLAG = 0, etiquetas y redshift válidos, y sin bandas vacías en el recorte; misma partición de entrenamiento, validación y prueba |
| Imagen | Bandas g, r y z en flujo físico, con `asinh(f / s_b)` (s_b: percentil 75 de \|f\| en entrenamiento) y estandarización por banda con entrenamiento |
| Entradas al predecir | Solo la imagen. En la variante en dos etapas, además, el redshift estimado desde la imagen, estandarizado con entrenamiento y unido al vector de la imagen antes de las cabezas |
| Salidas | Tres cabezas en paralelo, `Linear(d + k, 256) → ReLU → Dropout(0.1) → Linear(256, 1)`, con k = 0 o 1 variables escalares, sobre etiquetas estandarizadas |
| Aumentación | Giros de 90° y reflejos en entrenamiento; nada en validación y prueba |
| Criterio de selección | Promedio entre propiedades del MSE normalizado, (RMSE / σ_entrenamiento)²; se conserva la época con el menor valor en validación |
| Métricas | MAE, RMSE, sesgo y R² por propiedad, en su escala original, y el tiempo de entrenamiento |
| Semillas | 42, 7 y 2026; la semilla 42 representa al modelo en la tabla comparativa |
| Prueba | Se abre una sola vez, al final, con la elección registrada antes |

## 2. Parámetros

| Parámetro | Valor | Significado |
|---|---|---|
| `SEMILLAS` | 42, 7, 2026 | Semillas del entrenamiento; la 42 representa al modelo |
| `VARIANTES_ENTRADA` | solo imagen; imagen + redshift estimado | Las dos formas de usar el modelo que respetan la restricción |
| `EPOCAS` | 30 | Épocas de cada entrenamiento completo |
| `PRUEBAS_BUSQUEDA` | 30 | Configuraciones que prueba la búsqueda de hiperparámetros |
| `EPOCAS_BUSQUEDA` | 12 | Épocas máximas de cada prueba de la búsqueda (las poco prometedoras se detienen antes) |
| `FINALISTAS` | 3 | Mejores configuraciones de la búsqueda que se reentrenan con todas las épocas |
| `N_REMUESTREOS` | 2000 | Remuestreos *bootstrap* para los intervalos de confianza |
| `MARGEN` | 0.02, 0.02, 0.01 | Diferencia de RMSE que se considera despreciable en log M\*, log SFR y D4000 |

In [ ]:
import os
variable_entorno = os.environ.get          # las variables MA_* solo se usan en la prueba local reducida

SEMILLAS = tuple(int(s) for s in variable_entorno("MA_SEMILLAS", "42,7,2026").split(","))
EPOCAS = int(variable_entorno("MA_EPOCAS", "30"))
PRUEBAS_BUSQUEDA = int(variable_entorno("MA_PRUEBAS", "30"))
EPOCAS_BUSQUEDA = int(variable_entorno("MA_EPOCAS_BUSQUEDA", "12"))
FINALISTAS = int(variable_entorno("MA_FINALISTAS", "3"))
N_REMUESTREOS = int(variable_entorno("MA_REMUESTREOS", "2000"))
LIMITE_GALAXIAS = int(variable_entorno("MA_LIMITE", "0"))           # 0 = todas las galaxias
PREENTRENADO = variable_entorno("MA_PREENTRENADO", "1") == "1"
LADO_ENTRADA = int(variable_entorno("MA_LADO", "300"))
EXTRACTOR_BASE = variable_entorno("MA_EXTRACTOR", "convnext_tiny.fb_in22k_ft_in1k")
PESOS_CANDIDATOS = tuple(variable_entorno("MA_PESOS_CANDIDATOS",
                                          "convnext_tiny.fb_in22k_ft_in1k,convnext_tiny.fb_in1k,convnext_tiny.dinov3_lvd1689m").split(","))

from pathlib import Path
BASE_DRIVE = Path(variable_entorno("MA_BASE", "/content/drive/MyDrive"))
CARPETA_RESULTADOS = "resultados_modelos_alternativos"
NOMBRE_MODELO = "modelo3_convnext_tiny_imagen"
OBJETIVOS = ("log_mstar", "log_sfr", "d4000")
VARIANTES_ENTRADA = {"solo imagen": (), "imagen + redshift estimado": ("redshift_predicho",)}
NOMBRES = {"log_mstar": "log M* [dex]", "log_sfr": "log SFR [dex]", "d4000": "D4000"}
CORTO = {"log_mstar": "M*", "log_sfr": "SFR", "d4000": "D4000"}
MARGEN = {"M*": 0.02, "SFR": 0.02, "D4000": 0.01}
CORTE_SSFR = -10.8                                                   # frontera entre galaxias formadoras y apagadas
ENTRADA_BASE = {"representacion": "grz_estandarizado", "lado": LADO_ENTRADA, "modo_lado": "recorte"}

## 3. Entorno

Descargamos el código del repositorio del equipo, de modo que cada resultado quede ligado a una versión concreta, e instalamos las bibliotecas que Colab no incluye: `timm` (modelos preentrenados) y `optuna` (búsqueda de hiperparámetros). Los datos y los resultados están en Google Drive. Este cuaderno necesita una GPU; recomendamos una A100.

In [ ]:
import sys, json, time, subprocess, platform, warnings
from pathlib import Path
from dataclasses import replace

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
CARPETA_REPOSITORIO = Path(variable_entorno("MA_CARPETA_REPOSITORIO", "/content/InferenciaPropiedadesFisicas"))
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=1.0.20", "optuna>=4", "hickle"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    if not CARPETA_REPOSITORIO.exists():
        subprocess.run(["git", "clone", "-q", URL_REPOSITORIO, str(CARPETA_REPOSITORIO)], check=True)
    subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "pull", "-q"], check=True)
sys.path.insert(0, str(CARPETA_REPOSITORIO))
COMMIT = subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, timm, optuna
import matplotlib.pyplot as plt
from IPython.display import display

from src.modelos_alternativos import arquitecturas, busqueda, evaluacion, experimento, interpretacion
from src.modelos_alternativos.arquitecturas import EspecificacionModelo
from src.modelos_alternativos.contexto import preparar_contexto
from src.modelos_alternativos.entrenamiento import ConfiguracionEntrenamiento
from src.katachi_fits.diagnostico import radio_mitad_luz

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
pd.set_option("display.precision", 4, "display.max_columns", 40, "display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
print("Python", platform.python_version(), "· commit", COMMIT[:12], "· timm", timm.__version__, "· optuna", optuna.__version__,
      "·", DISPOSITIVO, torch.cuda.get_device_name(0) if DISPOSITIVO.type == "cuda" else "")

## 4. Datos

Preparamos la muestra y las transformaciones con las reglas del protocolo común. Todos los parámetros (la escala de compresión de cada banda y las medias y desviaciones de la imagen y de las etiquetas) se calculan solo con las galaxias de entrenamiento y se aplican sin cambios a validación y prueba. Los conteos de la muestra se calculan aquí, no se imponen.

In [ ]:
CONTEXTO = preparar_contexto(BASE_DRIVE, CARPETA_RESULTADOS, OBJETIVOS, limite_galaxias=LIMITE_GALAXIAS)
CARPETA_MODELO = CONTEXTO.guardia.mkdir(BASE_DRIVE / CARPETA_RESULTADOS / NOMBRE_MODELO)
MUESTRA, INDICES = CONTEXTO.muestra, CONTEXTO.indices
TIPO_GALAXIA = np.where(MUESTRA.log_ssfr < CORTE_SSFR, "apagada", "formadora")
display(pd.crosstab(MUESTRA.particion, pd.Series(TIPO_GALAXIA, name="tipo de galaxia", index=MUESTRA.index), margins=True,
                    margins_name="total").loc[["Train", "Validation", "Test", "total"]])
if len(CONTEXTO.excluidas):
    display(CONTEXTO.excluidas.groupby(["particion", "motivo"]).size().rename("galaxias").reset_index())
t = CONTEXTO.transformacion_grz
display(pd.DataFrame({"s_b (percentil 75 de |f|)": t.escala_asinh, "media tras asinh": t.media, "desviación tras asinh": t.desviacion},
                     index=["g", "r", "z"]).round(5))
CONTEXTO.guardia.write_json(CARPETA_MODELO / "contexto_datos.json", CONTEXTO.resumen(), allow_replace=True)
RADIO_MITAD_LUZ = radio_mitad_luz(CONTEXTO.imagenes, banda=1)          # en píxeles de la imagen de 300 px, banda r
print(f"Radio de mitad de luz (banda r): mediana {np.median(RADIO_MITAD_LUZ):.1f} px")

Las galaxias de la muestra se ven así a la entrada de la red. La imagen muestra los tres canales estandarizados (z, r y g como rojo, verde y azul) en el recorte central de 300 × 300 píxeles, que es la entrada de ConvNeXt. El círculo marca el radio que encierra la mitad de la luz.

In [ ]:
PREPARADOR_VISTA = experimento.crear_preparador(CONTEXTO, ENTRADA_BASE)
entrenamiento = MUESTRA.iloc[INDICES["Train"]]
ejemplos = [entrenamiento.index[np.argmin(np.abs(entrenamiento.log_mstar.to_numpy() - q))]
            for q in np.quantile(entrenamiento.log_mstar, np.linspace(0.05, 0.95, 6))]
fig, ejes = plt.subplots(1, 6, figsize=(15, 2.9))
for eje, i in zip(ejes, ejemplos):
    x = PREPARADOR_VISTA(torch.from_numpy(CONTEXTO.imagenes[i:i + 1].astype(np.float32)))[0].numpy()[[2, 1, 0]]
    if PREPARADOR_VISTA.representacion == "rgb_lupton":
        x = x[[2, 1, 0]]
    rgb = np.transpose(x, (1, 2, 0)); rgb = np.clip((rgb - np.percentile(rgb, 1)) / (np.percentile(rgb, 99.7) - np.percentile(rgb, 1)), 0, 1)
    eje.imshow(rgb, origin="lower")
    escala = x.shape[-1] / 300 if PREPARADOR_VISTA.modo_lado == "reescala" else 1
    eje.add_patch(plt.Circle(((x.shape[-1] - 1) / 2,) * 2, RADIO_MITAD_LUZ[i] * escala, fill=False, color="w", lw=0.8, ls="--"))
    g = MUESTRA.loc[i]
    eje.set_title(f"log M* {g.log_mstar:.1f} · log SFR {g.log_sfr:.1f}\nD4000 {g.d4000:.2f} · z {g.redshift:.3f}", fontsize=7)
    eje.set_xticks([]); eje.set_yticks([])
plt.suptitle(f"Entrada de la red ({x.shape[-1]} × {x.shape[-1]} px)", y=1.04); plt.show()

## 5. Redshift estimado desde la imagen

La variante en dos etapas necesita un redshift que se pueda obtener de la imagen. Lo estimamos con un modelo sencillo: un extractor congelado de DINOv3 (ViT-B/16, preentrenado con imágenes web) que convierte cada imagen en un vector, y encima las mismas cabezas de regresión del protocolo, entrenadas para predecir log₁₀ z. Usamos el logaritmo porque los errores de distancia son relativos: equivocarse por 0.01 importa mucho más a z = 0.02 que a z = 0.12.

Para que la segunda etapa aprenda con estimaciones del mismo nivel de error que tendrá con galaxias nuevas, las construimos con cuidado:

- **Galaxias de entrenamiento: predicciones fuera de partición** (*cross-fitting*; Wolpert 1992). Dividimos las galaxias de entrenamiento en 5 partes y estimamos cada parte con un modelo entrenado con las otras 4. Así ninguna galaxia recibe una estimación de un modelo que la vio. Si no lo hiciéramos, las estimaciones de entrenamiento serían casi perfectas, la segunda etapa aprendería a confiar en ellas y fallaría con galaxias nuevas.
- **Galaxias de validación y de prueba:** las estima un modelo entrenado con todas las galaxias de entrenamiento.
- **Época de cada modelo:** se elige con un 10 % de sus propias galaxias de ajuste, nunca con las galaxias que se van a estimar ni con validación.
- **Sin información de validación ni de prueba:** el redshift medido de esas galaxias no interviene en ningún ajuste. Calcular su estimación solo requiere sus imágenes, así que no equivale a abrir el conjunto de prueba; su calidad en prueba se reporta en la sección de evaluación final.

La estimación se guarda en Drive y la comparten todos los modelos alternativos, de modo que todos reciben exactamente el mismo redshift estimado.

In [ ]:
from src.modelos_alternativos.representaciones import ConfiguracionCabeza
from src.modelos_alternativos.variable_predicha import metricas_redshift, predecir_variable_desde_imagen

EXTRACTOR_ETAPA1 = variable_entorno("MA_EXTRACTOR_ETAPA1", "vit_base_patch16_dinov3.lvd1689m")
LADO_ETAPA1 = int(variable_entorno("MA_LADO_ETAPA1", "288"))
ESPEC_ETAPA1 = EspecificacionModelo(nombre_extractor=EXTRACTOR_ETAPA1, preentrenado=PREENTRENADO, lado_entrada=LADO_ETAPA1,
                                    estrategia_ajuste="congelado", agregacion_tokens="cls_y_promedio")
ENTRADA_ETAPA1 = {"representacion": "grz_estandarizado", "lado": LADO_ETAPA1, "modo_lado": "recorte"}
CABEZA_ETAPA1 = ConfiguracionCabeza(unidades_ocultas=256, dropout=0.1, tasa_aprendizaje=1e-3, decaimiento_pesos=1e-2,
                                    epocas=int(variable_entorno("MA_EPOCAS_ETAPA1", "300")), tamano_lote=256, perdida="huber",
                                    semilla=SEMILLAS[0])
TABLA_REDSHIFT = predecir_variable_desde_imagen(CONTEXTO, BASE_DRIVE / CARPETA_RESULTADOS / "etapa1_redshift_desde_imagen", "redshift",
                                                "redshift_predicho", ESPEC_ETAPA1, ENTRADA_ETAPA1, CABEZA_ETAPA1, DISPOSITIVO,
                                                n_particiones=5, fraccion_interna=0.1, transformacion="log10", semilla=SEMILLAS[0])
filas = []
for particion, origen in (("Train", "fuera de partición (5 partes)"), ("Validation", "modelo con todo entrenamiento")):
    idx = INDICES[particion]
    filas.append({"partición": particion, "origen": origen,
                  **metricas_redshift(MUESTRA.redshift.to_numpy()[idx], MUESTRA.redshift_predicho.to_numpy()[idx])})
CALIDAD_REDSHIFT = pd.DataFrame(filas).set_index("partición")
display(CALIDAD_REDSHIFT.round(4))

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for particion, color in (("Train", "0.6"), ("Validation", "#264653")):
    idx = INDICES[particion]
    z, zp = MUESTRA.redshift.to_numpy()[idx], MUESTRA.redshift_predicho.to_numpy()[idx]
    ejes[0].scatter(z, zp, s=4, alpha=0.4, color=color, label="entrenamiento (fuera de partición)" if particion == "Train" else "validación")
    dz = (zp - z) / (1 + z)
    orden = np.argsort(z); ventana = max(20, len(z) // 15)
    mediana = pd.Series(dz[orden]).rolling(ventana, center=True).median()
    ejes[1].plot(z[orden], mediana, color=color)
    ejes[2].hist(np.log10(zp / z), bins=50, histtype="step", density=True, color=color, lw=1.5)
limites = [MUESTRA.redshift.min() * 0.9, MUESTRA.redshift.max() * 1.1]
ejes[0].plot(limites, limites, "k-", lw=0.8); ejes[0].set(xscale="log", yscale="log", xlabel="redshift medido", ylabel="redshift estimado",
                                                          title="Estimación desde la imagen"); ejes[0].legend(fontsize=7)
ejes[1].axhline(0, color="k", lw=0.6); ejes[1].set(xscale="log", xlabel="redshift medido", ylabel="mediana de Δz / (1 + z)",
                                                   title="Sesgo según la distancia")
ejes[2].set(xlabel="log₁₀(z estimado / z medido)", ylabel="densidad", title="Error relativo")
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
# Utilidades para resumir corridas con las métricas del protocolo común.
def metricas_corrida(corrida, particion="Validation", tta=False):
    prediccion = experimento.predicciones_originales(CONTEXTO, corrida, particion, DISPOSITIVO, tta=tta)
    return experimento.tabla_corrida(CONTEXTO, prediccion, particion, corrida=corrida.identificador), prediccion

def resumen_corridas(corridas, etiqueta, particion="Validation"):
    # Media y desviación entre semillas de cada métrica; fila ancha por modelo.
    tablas = pd.concat([metricas_corrida(r, particion)[0].assign(semilla=r.resumen["configuracion"]["semilla"]) for r in corridas])
    variables = corridas[0].variables_escalares
    fila = {"modelo": etiqueta, "entradas": "solo imagen" if not variables else "imagen + " + ", ".join(variables).replace("_", " "),
            "solo_imagen_al_predecir": "sí", "semillas": len(corridas),
            "MSE_normalizado_medio": tablas.groupby("semilla").MSE_normalizado.mean().mean(),
            "desviacion_entre_semillas": tablas.groupby("semilla").MSE_normalizado.mean().std(ddof=1) if len(corridas) > 1 else np.nan}
    for objetivo, g in tablas.groupby("objetivo", sort=False):
        for m in ("RMSE", "MAE", "sesgo", "R2"):
            fila[f"{m} {CORTO[objetivo]}"] = g[m].mean()
    fila["minutos_entrenamiento"] = np.mean([r.resumen["minutos_entrenamiento"] for r in corridas])
    fila["parametros_entrenables_M"] = corridas[0].resumen["parametros_entrenables"] / 1e6
    return fila, tablas

## 6. Modelo base

El modelo base usa los pesos de ConvNeXt-Tiny preentrenados con ImageNet-22k y ajustados después con ImageNet-1k, que son los que mejor se transfieren según sus autores. Las tres bandas g, r y z entran en el lugar de los canales R, G y B, sin modificar la primera capa. El vector de 768 componentes que produce el extractor pasa a las tres cabezas del protocolo común; en la variante en dos etapas, antes se le une el redshift estimado.

La receta de entrenamiento sigue la de los autores de ConvNeXt para ajuste fino:

| Elemento | Valor | Motivo |
|---|---|---|
| Optimizador | AdamW, *weight decay* 0.05 | Regularización desacoplada de la tasa, estándar en ConvNeXt |
| Tasa del extractor | 1 × 10⁻⁴, con decaimiento por capa de 0.8 | Las capas iniciales, más genéricas, cambian menos que las finales |
| Tasa de las cabezas | 1 × 10⁻³ | Las capas nuevas aprenden más rápido que el extractor preentrenado |
| Programa | 2 épocas de calentamiento y descenso en coseno | Evita cambios bruscos al inicio, cuando las cabezas todavía son aleatorias |
| *Stochastic depth* | 0.1 | Regularización que omite bloques al azar durante el entrenamiento |
| Pérdida | Huber con δ = 1 sobre etiquetas estandarizadas | La misma del modelo 1 |
| Lote y precisión | 32 imágenes, bfloat16 | Aprovecha la A100 sin afectar la precisión de las predicciones |

Entrenamos esta configuración con las dos variantes de entrada y las tres semillas. Si un entrenamiento ya está en Drive, se recupera en lugar de repetirse. La comparación entre las dos variantes, sobre las mismas galaxias y con las mismas semillas, mide directamente si el redshift estimado ayuda.

In [ ]:
ESPECIFICACION_BASE = EspecificacionModelo(nombre_extractor=EXTRACTOR_BASE, preentrenado=PREENTRENADO, tasa_drop_path=0.1)
CONFIGURACION_BASE = ConfiguracionEntrenamiento(epocas=EPOCAS, epocas_calentamiento=2, tamano_lote=32, tasa_extractor=1e-4,
                                                tasa_cabeza=1e-3, decaimiento_pesos=0.05, decaimiento_por_capa=0.8, perdida="huber")
CORRIDAS_BASE, FILAS_BASE, TABLAS_BASE = {}, {}, []
for variante, variables in VARIANTES_ENTRADA.items():
    CORRIDAS_BASE[variante] = [experimento.ejecutar_corrida(CONTEXTO, CARPETA_MODELO, "base", ESPECIFICACION_BASE,
                                                            replace(CONFIGURACION_BASE, semilla=s), ENTRADA_BASE, DISPOSITIVO, variables)
                               for s in SEMILLAS]
    FILAS_BASE[variante], tabla = resumen_corridas(CORRIDAS_BASE[variante], f"3 · ConvNeXt-Tiny base, {variante}")
    TABLAS_BASE.append(tabla.assign(variante=variante))
TABLAS_BASE = pd.concat(TABLAS_BASE)
display(TABLAS_BASE.pivot_table(index=["variante", "objetivo"], columns="semilla", values="RMSE").round(4))
for variante, fila in FILAS_BASE.items():
    print(f"{variante}: MSE normalizado medio {fila['MSE_normalizado_medio']:.4f} ± {fila['desviacion_entre_semillas']:.4f}"
          f" · {fila['minutos_entrenamiento']:.1f} min por entrenamiento")

# ¿Ayuda el redshift estimado? Diferencia pareada sobre las mismas galaxias y semillas (negativo = ayuda)
y_validacion = CONTEXTO.etiquetas("Validation")
filas = []
for solo, dos_etapas in zip(*CORRIDAS_BASE.values()):
    ps = experimento.predicciones_originales(CONTEXTO, solo, "Validation", DISPOSITIVO)
    pd2 = experimento.predicciones_originales(CONTEXTO, dos_etapas, "Validation", DISPOSITIVO)
    for k, t in enumerate(OBJETIVOS):
        d, lo, hi = evaluacion.diferencia_pareada_rmse(y_validacion[:, k], pd2[:, k], ps[:, k], n_remuestreos=N_REMUESTREOS)
        filas.append({"semilla": solo.resumen["configuracion"]["semilla"], "propiedad": CORTO[t],
                      "ΔRMSE (dos etapas − solo imagen)": d, "IC 95 %": f"[{lo:+.3f}, {hi:+.3f}]",
                      "veredicto": evaluacion.veredicto(lo, hi, MARGEN[CORTO[t]])})
display(pd.DataFrame(filas).set_index(["propiedad", "semilla"]).round(4))

_Se completa con los resultados de la corrida._

## 7. Ajuste de hiperparámetros

Buscamos una configuración mejor con Optuna (Akiba et al. 2019). El muestreo bayesiano TPE propone cada nueva configuración a partir de los resultados de las anteriores, y la poda por *successive halving* detiene las pruebas que, en la misma época, van peor que la mayoría; así, el presupuesto de GPU se concentra en las configuraciones prometedoras. Cada prueba entrena como máximo 12 épocas con la semilla 42 y se califica con el mismo criterio de validación que el resto del avance. La variante de entrada también es un hiperparámetro, de modo que la búsqueda decide si conviene usar el redshift estimado.

| Hiperparámetro | Espacio de búsqueda | Qué controla |
|---|---|---|
| Variante de entrada | solo imagen; imagen + redshift estimado | Si el modelo recibe la estimación de la sección 5 |
| Tasa de aprendizaje del extractor | 2 × 10⁻⁵ a 5 × 10⁻⁴ (logarítmica) | Cuánto se modifican los pesos preentrenados |
| Tasa de las cabezas | 1 a 20 veces la del extractor (logarítmica) | Qué tan rápido aprenden las capas nuevas |
| *Weight decay* | 10⁻³ a 0.2 (logarítmica) | Regularización de los pesos |
| Decaimiento por capa | 0.6 a 1.0 | Cuánto menos cambian las capas iniciales que las finales |
| *Stochastic depth* | 0 a 0.3 | Regularización por omisión de bloques |
| Pesos de inicio | ImageNet-22k, ImageNet-1k o DINOv3 (destilados) | Qué representación preentrenada se aprovecha mejor |
| Pérdida | Huber o MSE | Sensibilidad a errores grandes |
| Épocas de calentamiento | 1 a 3 | Estabilidad al inicio |

El estudio se guarda en Drive después de cada prueba: si la sesión se desconecta, al volver a ejecutar la celda continúa donde quedó.

In [ ]:
def proponer(prueba):
    return {"variante_entrada": prueba.suggest_categorical("variante_entrada", list(VARIANTES_ENTRADA)),
            "tasa_extractor": prueba.suggest_float("tasa_extractor", 2e-5, 5e-4, log=True),
            "factor_tasa_cabeza": prueba.suggest_float("factor_tasa_cabeza", 1.0, 20.0, log=True),
            "decaimiento_pesos": prueba.suggest_float("decaimiento_pesos", 1e-3, 0.2, log=True),
            "decaimiento_por_capa": prueba.suggest_float("decaimiento_por_capa", 0.6, 1.0),
            "tasa_drop_path": prueba.suggest_float("tasa_drop_path", 0.0, 0.3),
            "pesos": prueba.suggest_categorical("pesos", list(PESOS_CANDIDATOS)),
            "perdida": prueba.suggest_categorical("perdida", ["huber", "mse"]),
            "epocas_calentamiento": prueba.suggest_int("epocas_calentamiento", 1, 3)}

def configuracion_desde(valores, epocas, semilla):
    espec = replace(ESPECIFICACION_BASE, nombre_extractor=str(valores["pesos"]), tasa_drop_path=float(valores["tasa_drop_path"]))
    config = replace(CONFIGURACION_BASE, epocas=epocas, semilla=semilla, tasa_extractor=float(valores["tasa_extractor"]),
                     tasa_cabeza=float(valores["tasa_extractor"] * valores["factor_tasa_cabeza"]),
                     decaimiento_pesos=float(valores["decaimiento_pesos"]), decaimiento_por_capa=float(valores["decaimiento_por_capa"]),
                     perdida=str(valores["perdida"]), epocas_calentamiento=int(min(int(valores["epocas_calentamiento"]), max(1, epocas - 1))))
    return espec, config, VARIANTES_ENTRADA[str(valores["variante_entrada"])]

def evaluar(valores, al_terminar_epoca):
    espec, config, variables = configuracion_desde(valores, EPOCAS_BUSQUEDA, SEMILLAS[0])
    corrida = experimento.ejecutar_corrida(CONTEXTO, CARPETA_MODELO, "busqueda", espec, config, ENTRADA_BASE, DISPOSITIVO, variables,
                                           al_terminar_epoca=al_terminar_epoca, guardar=False, registro=lambda *_: None)
    valor = corrida.resumen["mejor_criterio_validacion"]
    del corrida; torch.cuda.empty_cache()
    return valor

inicio = time.time()
respaldo = lambda origen, destino: CONTEXTO.guardia.write(destino, lambda f: Path(f).write_bytes(Path(origen).read_bytes()), allow_replace=True)
ESTUDIO = busqueda.ejecutar_busqueda(f"{NOMBRE_MODELO}{'_lim' + str(LIMITE_GALAXIAS) if LIMITE_GALAXIAS else ''}",
                                     CARPETA_MODELO / "busqueda_hiperparametros.db", proponer, evaluar, PRUEBAS_BUSQUEDA,
                                     semilla=SEMILLAS[0], epocas_minimas_antes_de_podar=3, guardar_respaldo=respaldo)
PRUEBAS = busqueda.tabla_pruebas(ESTUDIO)
print(f"Búsqueda: {len(PRUEBAS)} pruebas en {(time.time() - inicio) / 60:.0f} min · "
      f"{(PRUEBAS.estado == 'complete').sum()} terminadas, {(PRUEBAS.estado == 'pruned').sum()} podadas")
display(PRUEBAS.head(10).round(5))
CONTEXTO.guardia.write(CARPETA_MODELO / "pruebas_busqueda.csv", lambda f: PRUEBAS.to_csv(f, index=False), allow_replace=True)

In [ ]:
terminadas = PRUEBAS[PRUEBAS.estado == "complete"].sort_values("prueba")
mejor_base = min(f["MSE_normalizado_medio"] for f in FILAS_BASE.values())
fig, ejes = plt.subplots(1, 3, figsize=(16, 3.8))
for variante, g in terminadas.groupby("variante_entrada"):
    ejes[0].scatter(g.prueba, g.criterio_validacion, s=18, label=variante)
ejes[0].plot(terminadas.prueba, terminadas.criterio_validacion.cummin(), color="C3", label="mejor hasta el momento")
ejes[0].axhline(mejor_base, color="k", ls=":", lw=1, label="mejor configuración base")
ejes[0].set(xlabel="prueba", ylabel="criterio de validación", title="Historia de la búsqueda"); ejes[0].legend(fontsize=7)
for t in ESTUDIO.trials:
    if t.intermediate_values:
        e, v = zip(*sorted(t.intermediate_values.items()))
        ejes[1].plot(e, v, color="C3" if t.state.name == "PRUNED" else "C0", alpha=0.5, lw=0.8)
ejes[1].set(xlabel="época", ylabel="criterio de validación", yscale="log", title="Curvas por prueba (rojo: podadas)")
IMPORTANCIA_HIPERPARAMETROS = busqueda.importancia_hiperparametros(ESTUDIO)
if len(IMPORTANCIA_HIPERPARAMETROS):
    ejes[2].barh(IMPORTANCIA_HIPERPARAMETROS.index, IMPORTANCIA_HIPERPARAMETROS.values, color="#264653"); ejes[2].invert_yaxis()
ejes[2].set(xlabel="importancia (fANOVA)", title="Qué hiperparámetros pesan más")
plt.tight_layout(); plt.show()

numericos = ["tasa_extractor", "decaimiento_pesos", "decaimiento_por_capa", "tasa_drop_path"]
fig, ejes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
for eje, h in zip(ejes, numericos):
    for pesos, g in terminadas.groupby("pesos"):
        eje.scatter(g[h], g.criterio_validacion, s=18, label=pesos.split(".")[-1])
    eje.set(xlabel=h, xscale="log" if h in ("tasa_extractor", "decaimiento_pesos") else "linear")
ejes[0].set_ylabel("criterio de validación"); ejes[0].legend(fontsize=7, title="pesos"); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

Las pruebas de la búsqueda usan pocas épocas, así que su ordenamiento puede cambiar al entrenar más tiempo. Por eso reentrenamos las mejores configuraciones con todas las épocas y la semilla 42, elegimos la mejor por el criterio de validación y la entrenamos después con las otras dos semillas.

In [ ]:
mejores = PRUEBAS[PRUEBAS.estado == "complete"].head(FINALISTAS)
claves = ["variante_entrada", "tasa_extractor", "factor_tasa_cabeza", "decaimiento_pesos", "decaimiento_por_capa", "tasa_drop_path",
          "pesos", "perdida", "epocas_calentamiento"]
CORRIDAS_FINALISTAS, CONFIGURACIONES_FINALISTAS = [], []
for _, prueba_elegida in mejores.iterrows():
    espec, config, variables = configuracion_desde({k: prueba_elegida[k] for k in claves}, EPOCAS, SEMILLAS[0])
    CONFIGURACIONES_FINALISTAS.append((espec, config, variables))
    CORRIDAS_FINALISTAS.append(experimento.ejecutar_corrida(CONTEXTO, CARPETA_MODELO, f"finalista{int(prueba_elegida.prueba)}", espec,
                                                            config, ENTRADA_BASE, DISPOSITIVO, variables))
FINALISTAS_TABLA = pd.DataFrame([{"prueba": int(p.prueba), f"criterio en la búsqueda ({EPOCAS_BUSQUEDA} épocas)": p.criterio_validacion,
                                  "criterio con todas las épocas": r.resumen["mejor_criterio_validacion"],
                                  "mejor época": r.resumen["mejor_epoca"], **{k: p[k] for k in claves}}
                                 for (_, p), r in zip(mejores.iterrows(), CORRIDAS_FINALISTAS)])
display(FINALISTAS_TABLA.round(5))
posicion = int(np.argmin([r.resumen["mejor_criterio_validacion"] for r in CORRIDAS_FINALISTAS]))
CORRIDA_GANADORA = CORRIDAS_FINALISTAS[posicion]
espec_ganadora, config_ganadora, variables_ganadoras = CONFIGURACIONES_FINALISTAS[posicion]
VARIANTE_AJUSTADA = str(mejores.iloc[posicion].variante_entrada)
CORRIDAS_AJUSTADAS = [CORRIDA_GANADORA] + [
    experimento.ejecutar_corrida(CONTEXTO, CARPETA_MODELO, CORRIDA_GANADORA.identificador.split("_semilla")[0], espec_ganadora,
                                 replace(config_ganadora, semilla=s), ENTRADA_BASE, DISPOSITIVO, variables_ganadoras) for s in SEMILLAS[1:]]
FILA_AJUSTADA, _ = resumen_corridas(CORRIDAS_AJUSTADAS, f"3 · ConvNeXt-Tiny ajustado, {VARIANTE_AJUSTADA}")
base_misma_variante = CORRIDAS_BASE[VARIANTE_AJUSTADA]
print(f"Configuración ajustada ({VARIANTE_AJUSTADA}): {FILA_AJUSTADA['MSE_normalizado_medio']:.4f} ± {FILA_AJUSTADA['desviacion_entre_semillas']:.4f}"
      f" · base con la misma entrada: {FILAS_BASE[VARIANTE_AJUSTADA]['MSE_normalizado_medio']:.4f}")

filas = []
for base_corrida, ajustada in zip(base_misma_variante, CORRIDAS_AJUSTADAS):
    pb = experimento.predicciones_originales(CONTEXTO, base_corrida, "Validation", DISPOSITIVO)
    pa = experimento.predicciones_originales(CONTEXTO, ajustada, "Validation", DISPOSITIVO)
    for k, t in enumerate(OBJETIVOS):
        d, lo, hi = evaluacion.diferencia_pareada_rmse(y_validacion[:, k], pa[:, k], pb[:, k], n_remuestreos=N_REMUESTREOS)
        filas.append({"semilla": ajustada.resumen["configuracion"]["semilla"], "propiedad": CORTO[t], "ΔRMSE (ajustada − base)": d,
                      "IC 95 %": f"[{lo:+.3f}, {hi:+.3f}]", "veredicto": evaluacion.veredicto(lo, hi, MARGEN[CORTO[t]])})
display(pd.DataFrame(filas).set_index(["propiedad", "semilla"]).round(4))

_Se completa con los resultados de la corrida._

## 8. Comparación en validación

La tabla ordena los modelos por la métrica principal del protocolo común, el promedio del MSE normalizado, y muestra además el RMSE, el MAE, el sesgo y el R² de cada propiedad, el tiempo de entrenamiento y el número de parámetros entrenables. En los modelos con el extractor congelado, el tiempo incluye la extracción de las representaciones.

Como referencia incluimos los modelos que el equipo ya evaluó con el mismo protocolo, con la media de sus tres semillas:

- **Comparables:** las variantes F y G de ResNet50 (sin redshift) y el baseline del Avance 3 (cadena de ResNet50, solo imagen), recalculado sobre nuestras galaxias de validación. Igual que nuestros modelos, solo necesitan la imagen al predecir.
- **Techo, no comparables:** la variante E de ResNet50 y el SVR, que reciben el **redshift medido**. Ese dato no estará disponible en la práctica, así que no compiten; indican cuánto se pierde por no tenerlo.

In [ ]:
def _referencia(modelo, semillas, mse, rmse, mae, sesgo, r2, solo_imagen, fuente):
    fila = {"modelo": modelo, "entradas": "solo imagen" if solo_imagen == "sí" else "imagen + redshift medido",
            "solo_imagen_al_predecir": solo_imagen, "semillas": semillas, "MSE_normalizado_medio": mse, "fuente": fuente}
    for nombre, valores in (("RMSE", rmse), ("MAE", mae), ("sesgo", sesgo), ("R2", r2)):
        for corto, v in zip(("M*", "SFR", "D4000"), valores):
            fila[f"{nombre} {corto}"] = v
    return fila

REFERENCIAS = pd.DataFrame([
    _referencia("1 · ResNet50 multitarea, GRZ, Huber, sin redshift (variante G del equipo)", 3, 0.179510,
                (0.234420, 0.648439, 0.074901), (0.178870, 0.459307, 0.054264), (-0.018683, 0.005795, -0.000994),
                (0.872191, 0.689332, 0.898955), "sí", "reportado (1,261 galaxias)"),
    _referencia("1 · ResNet50 multitarea, GRZ, MSE, sin redshift (variante F del equipo)", 3, 0.178092,
                (0.236772, 0.641321, 0.074808), (0.181068, 0.458739, 0.054033), (-0.006764, 0.031536, -0.000373),
                (0.869582, 0.695999, 0.899214), "sí", "reportado (1,261 galaxias)"),
    _referencia("Techo · ResNet50 multitarea con redshift medido (variante E del equipo)", 3, 0.126881,
                (0.109265, 0.592216, 0.073610), (0.082881, 0.396620, 0.052947), (0.002940, -0.016601, 0.007759),
                (0.972233, 0.740901, 0.902414), "no", "reportado (1,261 galaxias)"),
    _referencia("Techo · SVR con redshift medido (modelo 2 del equipo, semilla 42)", 1, 0.442091,
                (0.266115, 0.982788, 0.161113), (0.199477, 0.759899, 0.125858), (-0.004865, 0.034490, -0.001860),
                (0.835294, 0.286476, 0.532613), "no", "reportado (1,261 galaxias)")])

def fila_baseline_avance3():
    # Cadena de ResNet50 del Avance 3 (solo imagen), recalculada sobre nuestras galaxias de validación.
    ruta = BASE_DRIVE / "resultados_katachi_fits" / "avance3" / "predicciones_B2_seed42.csv"
    if not ruta.exists():
        return None, None
    pred = pd.read_csv(ruta, dtype={"mangaid": str}).set_index("mangaid")
    ids = MUESTRA.mangaid.iloc[INDICES["Validation"]]
    p = pred.reindex(ids)[[f"pred_{t}" for t in OBJETIVOS]].to_numpy(float)
    if not np.isfinite(p).all():
        return None, None
    tabla = experimento.tabla_corrida(CONTEXTO, p, "Validation")
    fila = {"modelo": "Baseline del Avance 3 · cadena de ResNet50, solo imagen (semilla 42, con TTA)", "entradas": "solo imagen",
            "solo_imagen_al_predecir": "sí", "semillas": 1, "MSE_normalizado_medio": tabla.MSE_normalizado.mean(), "fuente": "recalculado"}
    for _, r in tabla.iterrows():
        for m in ("RMSE", "MAE", "sesgo", "R2"):
            fila[f"{m} {CORTO[r.objetivo]}"] = r[m]
    return fila, p

FILA_AVANCE3, PRED_AVANCE3_VALIDACION = fila_baseline_avance3()
filas = [*FILAS_BASE.values(), FILA_AJUSTADA]
COMPARATIVA = pd.concat([pd.DataFrame(filas).assign(fuente="este cuaderno"), REFERENCIAS,
                         pd.DataFrame([FILA_AVANCE3]) if FILA_AVANCE3 else None]).sort_values("MSE_normalizado_medio").reset_index(drop=True)
COMPARATIVA.index = COMPARATIVA.index + 1
columnas = ["modelo", "entradas", "solo_imagen_al_predecir", "MSE_normalizado_medio", "desviacion_entre_semillas", "RMSE M*", "RMSE SFR",
            "RMSE D4000", "MAE M*", "MAE SFR", "MAE D4000", "sesgo M*", "sesgo SFR", "sesgo D4000", "R2 M*", "R2 SFR", "R2 D4000",
            "minutos_entrenamiento", "parametros_entrenables_M", "semillas", "fuente"]
display(COMPARATIVA[[c for c in columnas if c in COMPARATIVA]].round(4))
CONTEXTO.guardia.write(CARPETA_MODELO / "comparativa_validacion.csv", lambda f: COMPARATIVA.to_csv(f, index=False), allow_replace=True)

fig, ejes = plt.subplots(1, 4, figsize=(17, 0.6 + 0.45 * len(COMPARATIVA)), sharey=True)
etiquetas = [m if len(m) < 62 else m[:59] + "…" for m in COMPARATIVA.modelo]
colores = ["#264653" if f == "este cuaderno" else ("#e9c46a" if s == "no" else "0.65")
           for f, s in zip(COMPARATIVA.fuente, COMPARATIVA.solo_imagen_al_predecir)]
for eje, columna in zip(ejes, ["MSE_normalizado_medio", "RMSE M*", "RMSE SFR", "RMSE D4000"]):
    eje.barh(etiquetas, COMPARATIVA[columna], color=colores); eje.set_xlabel(columna)
ejes[0].invert_yaxis()
plt.suptitle("Comparación en validación (oscuro: este cuaderno; gris: referencias solo con imagen; amarillo: techo con redshift medido)", y=1.02)
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
# Elección dentro de este cuaderno: la configuración con menor criterio medio entre semillas.
CANDIDATAS = {f"configuración base, {v}": (FILAS_BASE[v], CORRIDAS_BASE[v]) for v in VARIANTES_ENTRADA}
CANDIDATAS[f"configuración ajustada, {VARIANTE_AJUSTADA}"] = (FILA_AJUSTADA, CORRIDAS_AJUSTADAS)
NOMBRE_ELEGIDA = min(CANDIDATAS, key=lambda k: CANDIDATAS[k][0]["MSE_normalizado_medio"])
CORRIDAS_ELEGIDAS = CANDIDATAS[NOMBRE_ELEGIDA][1]
display(pd.DataFrame({k: {"MSE normalizado medio": v[0]["MSE_normalizado_medio"], "desviación entre semillas": v[0]["desviacion_entre_semillas"]}
                      for k, v in CANDIDATAS.items()}).T.round(4))
print("Configuración que representa al modelo 3:", NOMBRE_ELEGIDA)

## 9. Diagnóstico del modelo elegido

Revisamos el ajuste del modelo elegido con tres herramientas: la evolución del error por época en entrenamiento y validación, la comparación de las métricas entre las dos particiones y los residuos (predicción − etiqueta) frente al valor real y al redshift. Una brecha grande entre entrenamiento y validación indica sobreajuste; errores altos en ambas, junto con una pendiente negativa del residuo frente al valor real (regresión a la media), indican subajuste. El residuo frente al redshift medido muestra si el modelo todavía se equivoca según la distancia, el problema principal que encontramos en el Avance 3.

In [ ]:
CORRIDA_ELEGIDA = CORRIDAS_ELEGIDAS[0]
historial = CORRIDA_ELEGIDA.historial
if "perdida_entrenamiento" not in historial:
    print("El modelo elegido usa el extractor congelado: solo la cabeza aprendió y no hay curvas de entrenamiento de la imagen.")
else:
  fig, ejes = plt.subplots(1, 4, figsize=(17, 3.4))
  ejes[0].plot(historial.epoca, historial.perdida_entrenamiento, label="pérdida de entrenamiento")
  ejes[0].plot(historial.epoca, historial.criterio_validacion, label="criterio de validación")
  ejes[0].axvline(CORRIDA_ELEGIDA.resumen["mejor_epoca"], color="k", ls=":", lw=1, label="época elegida")
  ejes[0].set(xlabel="época", yscale="log", title="Curvas de entrenamiento"); ejes[0].legend(fontsize=7)
  for k, (eje, t) in enumerate(zip(ejes[1:], OBJETIVOS)):
      eje.plot(historial.epoca, np.sqrt(historial[f"mse_validacion_salida{k}"]) * CONTEXTO.desviacion_etiquetas_entrenamiento[k], color="#264653")
      eje.set(xlabel="época", ylabel="RMSE en validación", title=NOMBRES[t])
  plt.tight_layout(); plt.show()

ajuste = pd.concat([metricas_corrida(CORRIDA_ELEGIDA, p)[0] for p in ("Train", "Validation")])
ajuste = ajuste.pivot_table(index="objetivo", columns="particion", values=["RMSE", "sesgo", "sigma_NMAD"]).reindex(list(OBJETIVOS))
ajuste[("cociente", "Validation / Train")] = ajuste[("RMSE", "Validation")] / ajuste[("RMSE", "Train")]
ajuste[("referencia", "error de la etiqueta")] = [MUESTRA[f"error_{t}"].median() for t in OBJETIVOS]
display(ajuste.round(4))

_Se completa con los resultados de la corrida._

In [ ]:
PRED_VALIDACION = experimento.predicciones_originales(CONTEXTO, CORRIDA_ELEGIDA, "Validation", DISPOSITIVO)
Y_VALIDACION = CONTEXTO.etiquetas("Validation")
z_validacion = MUESTRA.redshift.to_numpy()[INDICES["Validation"]]

def mediana_movil(x, r, n=10):
    bordes = np.quantile(x, np.linspace(0, 1, n + 1)); centro, med, p16, p84 = [], [], [], []
    for a, b in zip(bordes[:-1], bordes[1:]):
        s = (x >= a) & (x <= b)
        if s.sum() >= 10:
            centro.append(np.median(x[s])); med.append(np.median(r[s])); p16.append(np.percentile(r[s], 16)); p84.append(np.percentile(r[s], 84))
    return map(np.array, (centro, med, p16, p84))

fig, ejes = plt.subplots(3, 3, figsize=(14, 11))
for k, t in enumerate(OBJETIVOS):
    y, p = Y_VALIDACION[:, k], PRED_VALIDACION[:, k]
    lim = np.percentile(np.r_[y, p], [0.5, 99.5])
    ejes[0, k].hexbin(y, p, gridsize=45, cmap="viridis", mincnt=1, bins="log", extent=(*lim, *lim))
    ejes[0, k].plot(lim, lim, "k-", lw=0.8)
    m = evaluacion.metricas(y, p)
    ejes[0, k].text(0.03, 0.97, f"RMSE {m['RMSE']:.3f}\nsesgo {m['sesgo']:+.3f}\nσ_NMAD {m['sigma_NMAD']:.3f}", transform=ejes[0, k].transAxes,
                    va="top", fontsize=8, bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="0.7"))
    ejes[0, k].set(xlim=lim, ylim=lim, xlabel=f"{NOMBRES[t]} (etiqueta)", ylabel="predicción")
    for fila, x, nombre_x in ((1, y, f"{NOMBRES[t]} (etiqueta)"), (2, z_validacion, "redshift")):
        cx, med, p16, p84 = mediana_movil(x, p - y)
        ejes[fila, k].plot(cx, med, "o-", ms=3, color="#264653"); ejes[fila, k].fill_between(cx, p16, p84, color="#264653", alpha=0.2)
        ejes[fila, k].axhline(0, color="k", lw=0.6); ejes[fila, k].set(xlabel=nombre_x, ylabel="residuo")
plt.suptitle("Validación: predicción frente a etiqueta y mediana del residuo (franja: 68 % central)", y=1.0); plt.tight_layout(); plt.show()

tipo_validacion = TIPO_GALAXIA[INDICES["Validation"]]
display(evaluacion.metricas_por_grupo(Y_VALIDACION, PRED_VALIDACION, tipo_validacion, OBJETIVOS)
        .pivot(index="objetivo", columns="grupo", values=["RMSE", "sesgo"]).reindex(list(OBJETIVOS)).round(4))

_Se completa con los resultados de la corrida._

## 10. ¿Qué información usa el modelo?

Medimos la importancia de cada entrada como el aumento del error cuando el modelo deja de recibirla, en las galaxias de validación. Quitamos una banda a la vez y una región de la galaxia a la vez (anillos en unidades del radio de mitad de luz) y, si el modelo usa el redshift estimado, lo revolvemos entre galaxias. Las bandas y las regiones se quitan en la imagen original, poniendo su flujo en el nivel del cielo (0, porque las imágenes de DESI tienen el cielo restado), y después la imagen pasa por la misma preparación que usa el modelo. Las imágenes modificadas son distintas de las que el modelo vio al entrenar, así que la prueba indica qué información usa, no qué información existe. Con la misma prueba en todos los modelos podemos comparar cómo razonan.

In [ ]:
conjunto_validacion = experimento.conjunto_de(CONTEXTO, CORRIDA_ELEGIDA, "Validation", DISPOSITIVO)
radio_validacion = RADIO_MITAD_LUZ[INDICES["Validation"]]
base = CORRIDA_ELEGIDA.preparador
prediccion_base = CONTEXTO.a_escala_original(interpretacion.predecir_con_modificacion(CORRIDA_ELEGIDA.modelo, base, conjunto_validacion))
pruebas = [("banda", f"banda {nombre}", dict(banda=k)) for k, nombre in enumerate(("g", "r", "z"))]
pruebas += [("región", f"{a}–{b} R50" if np.isfinite(b) else f"> {a} R50", dict(anillo=(a, b), radio_mitad_luz_px=radio_validacion))
            for a, b in [(0, 0.5), (0.5, 1), (1, 2), (2, 4), (4, np.inf)]]
pruebas += [("variable escalar", f"{nombre} revuelto", dict(escalar_permutado=k, semilla=SEMILLAS[0]))
            for k, nombre in enumerate(CORRIDA_ELEGIDA.variables_escalares)]
filas = []
for tipo, nombre, argumentos in pruebas:
    p = CONTEXTO.a_escala_original(interpretacion.predecir_con_modificacion(CORRIDA_ELEGIDA.modelo, base, conjunto_validacion, **argumentos))
    for k, t in enumerate(OBJETIVOS):
        d, lo, hi = evaluacion.diferencia_pareada_rmse(Y_VALIDACION[:, k], p[:, k], prediccion_base[:, k], n_remuestreos=N_REMUESTREOS)
        filas.append({"tipo": tipo, "entrada quitada": nombre, "propiedad": CORTO[t], "aumento del RMSE": d, "lo": lo, "hi": hi})
IMPORTANCIA = pd.DataFrame(filas)
CONTEXTO.guardia.write(CARPETA_MODELO / "importancia_entradas.csv", lambda f: IMPORTANCIA.to_csv(f, index=False), allow_replace=True)
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for eje, t in zip(ejes, OBJETIVOS):
    d = IMPORTANCIA[IMPORTANCIA.propiedad == CORTO[t]]
    colores = d.tipo.map({"banda": "#2a9d8f", "región": "#264653", "variable escalar": "#e76f51"})
    eje.barh(d["entrada quitada"], d["aumento del RMSE"], xerr=[d["aumento del RMSE"] - d.lo, d.hi - d["aumento del RMSE"]],
             color=colores, capsize=2)
    eje.axvline(0, color="k", lw=0.6); eje.set(xlabel=f"aumento del RMSE · {NOMBRES[t]}")
ejes[0].invert_yaxis(); plt.suptitle("Importancia de cada entrada (verde: bandas; azul: regiones; naranja: redshift estimado)", y=1.02)
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

## 11. Evaluación en el conjunto de prueba

Con el modelo, su configuración y sus semillas ya fijados en validación, registramos la elección y abrimos el conjunto de prueba **una sola vez**. El registro guarda la configuración y una huella de las predicciones de validación; si el cuaderno se vuelve a ejecutar con otro modelo, avisa que esa evaluación ya no es independiente.

Reportamos las métricas del protocolo común con la semilla 42 y la media de las tres semillas, con y sin promedio de las 8 simetrías de cada imagen (*test-time augmentation*, TTA). Además comparamos, galaxia por galaxia, con el baseline del Avance 3 y con las predicciones publicadas de Katachi (Alfonzo et al. 2024) en las mismas galaxias.

In [ ]:
CORRIDAS_FINALES = CORRIDAS_ELEGIDAS
ELECCION = {"modelo": NOMBRE_MODELO, "descripcion": f'ConvNeXt-Tiny, {NOMBRE_ELEGIDA}', "corridas": [r.identificador for r in CORRIDAS_FINALES],
            "entradas": list(CORRIDAS_FINALES[0].variables_escalares), "commit": COMMIT}
experimento.registrar_eleccion_antes_de_prueba(CONTEXTO, CARPETA_MODELO / "registro_prueba.json", ELECCION,
                                               experimento.predicciones_originales(CONTEXTO, CORRIDAS_FINALES[0], "Validation", DISPOSITIVO))
Y_PRUEBA = CONTEXTO.etiquetas("Test")
tablas, PRED_PRUEBA = [], {}
for corrida in CORRIDAS_FINALES:
    semilla = corrida.resumen["configuracion"]["semilla"]
    for tta in (False, True):
        p = experimento.predicciones_originales(CONTEXTO, corrida, "Test", DISPOSITIVO, tta=tta)
        PRED_PRUEBA[(semilla, tta)] = p
        tablas.append(experimento.tabla_corrida(CONTEXTO, p, "Test", semilla=semilla, tta="sí" if tta else "no"))
    experimento.guardar_predicciones(CONTEXTO, CARPETA_MODELO, "Test", PRED_PRUEBA[(semilla, False)], f"predicciones_semilla{semilla}")
METRICAS_PRUEBA = pd.concat(tablas)
CONTEXTO.guardia.write(CARPETA_MODELO / "metricas_prueba.csv", lambda f: METRICAS_PRUEBA.to_csv(f, index=False), allow_replace=True)
resumen_prueba = (METRICAS_PRUEBA.groupby(["tta", "objetivo"], sort=False)[["RMSE", "MAE", "sesgo", "R2", "MSE_normalizado"]]
                  .agg(["mean", "std"]).round(4))
display(resumen_prueba)
for tta in ("no", "sí"):
    medio = METRICAS_PRUEBA[METRICAS_PRUEBA.tta == tta].groupby("semilla").MSE_normalizado.mean()
    print(f"MSE normalizado medio en prueba, TTA {tta}: {medio.mean():.4f} ± {medio.std(ddof=1) if len(medio) > 1 else float('nan'):.4f}")
if "redshift_predicho" in MUESTRA:
    idx = INDICES["Test"]
    print("Calidad del redshift estimado en prueba:",
          {k: round(v, 4) for k, v in metricas_redshift(MUESTRA.redshift.to_numpy()[idx], MUESTRA.redshift_predicho.to_numpy()[idx]).items()})

_Se completa con los resultados de la corrida._

In [ ]:
ids_prueba = MUESTRA.mangaid.iloc[INDICES["Test"]].to_numpy()
referencias_prueba = {}
ruta_avance3 = BASE_DRIVE / "resultados_katachi_fits" / "avance3" / "predicciones_prueba_B2_seed42.csv"
if ruta_avance3.exists():
    tabla = pd.read_csv(ruta_avance3, dtype={"mangaid": str}).set_index("mangaid")
    referencias_prueba["Baseline del Avance 3 (solo imagen)"] = tabla.reindex(ids_prueba)[[f"pred_{t}" for t in OBJETIVOS]].to_numpy(float)
columnas_katachi = ["pred_mstar", "pred_sfr", "pred_d4000"]
if set(columnas_katachi) <= set(MUESTRA.columns):
    referencias_prueba["Katachi publicado (imágenes SDSS)"] = MUESTRA.iloc[INDICES["Test"]][columnas_katachi].to_numpy(float)
filas = []
for nombre, referencia in referencias_prueba.items():
    validas = np.isfinite(referencia).all(1)
    for semilla in SEMILLAS:
        propia = PRED_PRUEBA.get((semilla, False))
        if propia is None:
            continue
        for k, t in enumerate(OBJETIVOS):
            d, lo, hi = evaluacion.diferencia_pareada_rmse(Y_PRUEBA[validas, k], propia[validas, k], referencia[validas, k],
                                                           n_remuestreos=N_REMUESTREOS)
            filas.append({"comparado con": nombre, "semilla": semilla, "propiedad": CORTO[t], "galaxias": int(validas.sum()),
                          "ΔRMSE (este modelo − referencia)": d, "IC 95 %": f"[{lo:+.3f}, {hi:+.3f}]",
                          "veredicto": evaluacion.veredicto(lo, hi, MARGEN[CORTO[t]])})
COMPARACION_PRUEBA = pd.DataFrame(filas)
if len(COMPARACION_PRUEBA):
    display(COMPARACION_PRUEBA.set_index(["comparado con", "propiedad", "semilla"]).round(4))

_Se completa con los resultados de la corrida._

## 12. Resumen para la tabla comparativa del avance

Guardamos en Drive la fila de este modelo con la métrica principal y las métricas complementarias en validación y en prueba, el tiempo de entrenamiento, el tiempo de inferencia y el tamaño del modelo. El cuaderno que reúne los seis modelos lee estos archivos.

In [ ]:
fila, _ = resumen_corridas(CORRIDAS_FINALES, f'3 · ConvNeXt-Tiny, {NOMBRE_ELEGIDA}')
prueba_semilla42 = METRICAS_PRUEBA[(METRICAS_PRUEBA.semilla == SEMILLAS[0]) & (METRICAS_PRUEBA.tta == "no")]
fila.update({"MSE_normalizado_prueba_semilla42": prueba_semilla42.MSE_normalizado.mean(),
             **{f"RMSE prueba {CORTO[r.objetivo]}": r.RMSE for r in prueba_semilla42.itertuples()},
             "ms_por_galaxia_inferencia": experimento.tiempo_inferencia_ms(CONTEXTO, CORRIDAS_FINALES[0], DISPOSITIVO),
             "parametros_totales_M": CORRIDAS_FINALES[0].resumen["parametros_totales"] / 1e6,
             "configuracion": json.dumps({"especificacion": CORRIDAS_FINALES[0].resumen["especificacion"],
                                          "configuracion": CORRIDAS_FINALES[0].resumen["configuracion"],
                                          "entrada": CORRIDAS_FINALES[0].resumen["entrada"],
                                          "variables_escalares": list(CORRIDAS_FINALES[0].variables_escalares)}, ensure_ascii=False)})
FILA_COMPARATIVA = pd.DataFrame([fila])
CONTEXTO.guardia.write(CARPETA_MODELO / "fila_comparativa.csv", lambda f: FILA_COMPARATIVA.to_csv(f, index=False), allow_replace=True)
display(FILA_COMPARATIVA.drop(columns=["configuracion"]).T.rename(columns={0: "valor"}))

## 13. Conclusiones

_Se completa con los resultados de la corrida._

## 14. Referencias

- Akiba, T., Sano, S., Yanase, T., Ohta, T. y Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *Proceedings of the 25th ACM SIGKDD*, 2623–2631.
- Alfonzo, J. P., Iyer, K. G., Akiyama, M., et al. (2024). Decoding the imprints of past star formation on present-day morphology in galaxies with interpretable CNNs. *The Astrophysical Journal*, 967, 152.
- Fang, G., Zhu, S., Xu, J., et al. (2025). An updated efficient galaxy morphology classification model based on ConvNeXt encoding with UMAP dimensionality reduction. arXiv:2512.15137.
- Huang, G., Sun, Y., Liu, Z., Sedra, D. y Weinberger, K. Q. (2016). Deep networks with stochastic depth. *ECCV*, 646–661.
- Li, L., Jamieson, K., Rostamizadeh, A., et al. (2020). A system for massively parallel hyperparameter tuning. *Proceedings of Machine Learning and Systems*, 2, 230–246.
- Liu, Z., Mao, H., Wu, C.-Y., Feichtenhofer, C., Darrell, T. y Xie, S. (2022). A ConvNet for the 2020s. *CVPR*, 11976–11986.
- Loshchilov, I. y Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*.
- Siméoni, O., Vo, H. V., Seitzer, M., et al. (2025). DINOv3. arXiv:2508.10104.
- Studer, S., Bui, T. B., Drescher, C., et al. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction*, 3, 392–413.
- Wolpert, D. H. (1992). Stacked generalization. *Neural Networks*, 5, 241–259.